# Notebook purpose

This notebook preserves the exploratory workflow from `euclidean.ipynb`.

Purpose: Explore yearwise party distance matrices and heatmaps.

Related production code/script: `src/analysis/distance.py`, `scripts/run_distance_analysis.py`.


In [8]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np # Often needed with seaborn/matplotlib
from tqdm.auto import tqdm
import torch
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import pdist, squareform

In [9]:
import os
import numpy as np
import pandas as pd
import umap
import matplotlib.pyplot as plt

# ---------- caminhos ----------
BASE       = "data/discursos_camara_2003_2025_limpos_Linq"
CSV_PATH   = f"{BASE}_com_id.csv"
EMB_PATH   = f"{BASE}_embeddings_raw.npy"
IDS_PATH   = f"{BASE}_ids.csv"
PLOT_DIR   = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)

# ---------- ler dados ----------
df   = pd.read_csv(CSV_PATH, dtype={"ano": int})
embs = np.load(EMB_PATH)
ids  = pd.read_csv(IDS_PATH)
print(f"✔️  Dados carregados: {len(df)} discursos, {embs.shape} embeddings, {len(ids)} IDs")
assert len(df) == len(embs) == len(ids), "Tamanhos inconsistentes!"

# ---------- descobrir anos ----------
anos_unicos = sorted(df["ano"].unique())
total_anos  = len(anos_unicos)

# ---------- partidos presentes em todos os anos ----------
partidos_full = (
    df.groupby("partido")["ano"]
      .nunique()
      .loc[lambda s: s == total_anos]
      .index
      .tolist()
)

cores_por_partido = {
    # Paleta de alto contraste para máxima diferenciação visual.
    "PT": "#E6194B",    # Vermelho
    "PSDB": "#3cb44b",  # Verde
    "MDB": "#4363d8",    # Azul
    "PP": "#911eb4",    # Roxo
    "PSB": "#f18e46",    # Laranja
    "PL": "#ffe119",    # Amarelo
    "PCdoB": "#971111",  # Marrom (Vinho)
    "PDT": "#42d4f4",    # Ciano
    "DEM": "#f032e6",    # Magenta
    "PSOL": "#F14545",   # Marrom (Sépia)
    "PTB": "#000075",    # Azul Marinho
    "REPUBLICANOS": "#28a745", # Verde (Escuro)
    "PFL": "#808000",    # Oliva
    "PSD": "#469990",    # Verde-azulado (Teal)
    "PV": "#bfef45",    # Verde-limão
    "NOVO": "#ff7300",   # Laranja (Escuro)
    "PSL": "#04787c",    
    "PSC": "#fabebe",    # Rosa (Salmão)
    "PRB": "#dcbeff",    # Lavanda
    "PPS": "#a9a9a9",    # Cinza (escuro)
    # Partidos com nomes antigos ou que se fundiram
    "PMDB": "#4363d8",  # Azul (mesma cor do MDB)
    "PR": "#ffe119",    # Amarelo (mesma cor do PL)
}

partidos_para_plotar = list(cores_por_partido.keys())
print(f"✔️  Plotando {len(partidos_para_plotar)} partidos definidos no dicionário.")

# ---------- subset + mean-pool ----------
# Filtra o DataFrame para incluir APENAS os partidos da lista
df_filtrado = df[df["partido"].isin(partidos_para_plotar)].copy()
grouped = (
    df_filtrado  # Usa o DataFrame já filtrado
    .groupby(["partido", "ano"])
    .apply(lambda g: embs[g["discurso_id"].values].mean(axis=0))
)
agg_df  = grouped.to_frame(name="vector").reset_index()
agg_mat = np.vstack(agg_df["vector"].values)

# A dictionary to hold the distance matrix for each year
DISTANCE_DIR = "distancias_por_ano"
os.makedirs(DISTANCE_DIR, exist_ok=True)

# A dictionary to hold the distance matrix for each year
distances_by_year = {}

# Group the aggregated dataframe by year
for year, year_df in agg_df.groupby("ano"):
    # Get the list of parties for the current year to use as labels
    parties = year_df["partido"].tolist()

    # Stack the embedding vectors for the current year into a matrix
    vectors = np.vstack(year_df["vector"].values)

    # Calculate the pairwise Euclidean distances
    distance_matrix = pdist(vectors, metric='euclidean')  # Using cosine distance for better semantic similarity

    # Convert to a square DataFrame
    distance_df = pd.DataFrame(
        squareform(distance_matrix),
        index=parties,
        columns=parties
    )

    # Store the DataFrame in the dictionary (optional, but good practice)
    distances_by_year[year] = distance_df

    # ---------- Salvar o DataFrame de distância em um arquivo CSV ----------
    output_path = os.path.join(DISTANCE_DIR, f"distancias_{year}.csv")
    distance_df.to_csv(output_path)

print(f"✔️ Distances for all years saved in the directory: {os.path.abspath(DISTANCE_DIR)}")

# --- Example: Print the distances for a specific year (optional) ---
example_year = 2003
if example_year in distances_by_year:
    print(f"\n--- Example for {example_year} ---")
    print(f"Euclidean distances between parties in {example_year}:\n")
    print(distances_by_year[example_year].round(4))


✔️  Dados carregados: 453211 discursos, (453211, 4096) embeddings, 453211 IDs
✔️  Plotando 22 partidos definidos no dicionário.
✔️ Distances for all years saved in the directory: /home/flaviossf/work/deputados/distancias_por_ano

--- Example for 2003 ---
Euclidean distances between parties in 2003:

         PCdoB      PDT      PFL     PMDB      PPS      PSB      PSC     PSDB  \
PCdoB   0.0000  11.4336  10.2494  10.8210  12.4269  10.4094  19.0700  10.4581   
PDT    11.4336   0.0000   8.3540   7.8488   9.8036   7.3218  18.2181   7.8716   
PFL    10.2494   8.3540   0.0000   7.6007   8.7294   8.0248  17.8267   4.8378   
PMDB   10.8210   7.8488   7.6007   0.0000   9.7678   7.2188  16.9614   7.8702   
PPS    12.4269   9.8036   8.7294   9.7678   0.0000   9.1926  19.4716   8.8242   
PSB    10.4094   7.3218   8.0248   7.2188   9.1926   0.0000  17.2953   7.4787   
PSC    19.0700  18.2181  17.8267  16.9614  19.4716  17.2953   0.0000  18.0305   
PSDB   10.4581   7.8716   4.8378   7.8702   8.8242 

/tmp/ipykernel_140671/513257333.py:71: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: embs[g["discurso_id"].values].mean(axis=0))


In [12]:
DISTANCE_DIR = "distancias_por_ano"  # Diretório onde os CSVs de distância estão
HEATMAP_DIR = "heatmaps_distancias"   # Novo diretório para salvar os heatmaps
os.makedirs(HEATMAP_DIR, exist_ok=True)

# --- Listar todos os arquivos CSV no diretório de distâncias ---
try:
    csv_files = [f for f in os.listdir(DISTANCE_DIR) if f.startswith("distancias_") and f.endswith(".csv")]
except FileNotFoundError:
    print(f"Erro: O diretório '{DISTANCE_DIR}' não foi encontrado. Certifique-se de que ele existe e contém os arquivos CSV.")
    csv_files = []

if not csv_files:
    print(f"Nenhum arquivo CSV de distância encontrado em '{DISTANCE_DIR}'.")
else:
    print(f"Encontrados {len(csv_files)} arquivos CSV de distância para processar.")

csv_files.sort()  # Ordenar os arquivos para garantir que sejam processados na ordem correta

# --- Gerar e salvar heatmaps ---
for csv_file in csv_files:
    try:
        # Extrair o ano do nome do arquivo
        # Espera-se um formato como "distancias_2003.csv"
        year_str = csv_file.replace("distancias_", "").replace(".csv", "")
        year = int(year_str) # Converte para inteiro para uso no título, se necessário

        # Caminho completo para o arquivo CSV
        file_path = os.path.join(DISTANCE_DIR, csv_file)

        # Ler o DataFrame de distância, usando a primeira coluna como índice
        distance_df = pd.read_csv(file_path, index_col=0)

        # --- Criar o heatmap ---
        plt.figure(figsize=(12, 10)) # Ajuste o tamanho conforme necessário
        sns.heatmap(
            distance_df,
            annot=True,          # Mostrar os valores de distância nas células
            fmt=".2f",           # Formatar os valores com 2 casas decimais
            cmap="viridis_r",    # Escolha um mapa de cores (ex: 'viridis_r', 'coolwarm', 'YlGnBu')
                                 # '_r' inverte o mapa de cores (menores distâncias -> cores mais "quentes" se desejar)
            linewidths=.5,       # Linhas entre as células
            cbar_kws={'label': 'Distância Euclidiana'} # Label da barra de cores
        )
        plt.title(f"Heatmap de Distâncias Euclidianas entre Partidos - {year}", fontsize=15)
        plt.xticks(rotation=45, ha="right", fontsize=10) # Rotacionar labels do eixo X para melhor visualização
        plt.yticks(rotation=0, fontsize=10)             # Labels do eixo Y
        plt.tight_layout() # Ajusta o plot para caber tudo direitinho

        # Salvar o heatmap
        heatmap_filename = f"heatmap_distancias_{year}.png"
        heatmap_save_path = os.path.join(HEATMAP_DIR, heatmap_filename)
        plt.savefig(heatmap_save_path, dpi=300) # dpi para melhor resolução
        plt.close() # Fechar a figura para liberar memória

        print(f"✔️ Heatmap salvo para o ano {year} em: {heatmap_save_path}")

    except ValueError:
        print(f"Erro ao processar o arquivo {csv_file}: Não foi possível converter o ano para inteiro. Verifique o nome do arquivo.")
    except FileNotFoundError:
        print(f"Erro: O arquivo {csv_file} não foi encontrado no diretório {DISTANCE_DIR}.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado ao processar {csv_file}: {e}")

if csv_files:
    print(f"\n🎉 Processamento de heatmaps concluído. Arquivos salvos em: {os.path.abspath(HEATMAP_DIR)}")


Encontrados 23 arquivos CSV de distância para processar.
✔️ Heatmap salvo para o ano 2003 em: heatmaps_distancias/heatmap_distancias_2003.png
✔️ Heatmap salvo para o ano 2004 em: heatmaps_distancias/heatmap_distancias_2004.png
✔️ Heatmap salvo para o ano 2005 em: heatmaps_distancias/heatmap_distancias_2005.png
✔️ Heatmap salvo para o ano 2006 em: heatmaps_distancias/heatmap_distancias_2006.png
✔️ Heatmap salvo para o ano 2007 em: heatmaps_distancias/heatmap_distancias_2007.png
✔️ Heatmap salvo para o ano 2008 em: heatmaps_distancias/heatmap_distancias_2008.png
✔️ Heatmap salvo para o ano 2009 em: heatmaps_distancias/heatmap_distancias_2009.png
✔️ Heatmap salvo para o ano 2010 em: heatmaps_distancias/heatmap_distancias_2010.png
✔️ Heatmap salvo para o ano 2011 em: heatmaps_distancias/heatmap_distancias_2011.png
✔️ Heatmap salvo para o ano 2012 em: heatmaps_distancias/heatmap_distancias_2012.png
✔️ Heatmap salvo para o ano 2013 em: heatmaps_distancias/heatmap_distancias_2013.png
✔️ Heatm